# ParSNIP-first transient classifier

This notebook trains the first reproducible seven-class baseline on the simulated warp-template sample. The first run deliberately uses exact simulated redshift (`truth_z`), so its score is an upper-bound baseline rather than an estimate of performance with uncertain host redshifts.

The split is made by a physical provenance group, not by random light curves. All objects generated from one `template_key` therefore stay together. Set `SPLIT_STRATEGY = "basis_sn"` for the stricter alternative, which holds out every warp derived from the same basis supernova together. This prevents *leakage*: a model cannot score well merely by recognizing close variants of a training template. Fold 0 is test, fold 1 is validation, and folds 2–9 are training. Validation is used for model choices; test is touched only after the experiment is frozen.

An *epoch* is one complete optimization pass as defined by ParSNIP. ParSNIP augments small datasets internally, so a smoke epoch can contain repeated transformed views. The generative ParSNIP stage is label-free; class weights apply to the supervised LightGBM stage so that every final class has equal total training influence. These balanced probabilities are useful for comparing classes, but they are not calibrated to real survey class frequencies.

In [ ]:
# Import the shared workflow and resolve paths from either notebook launch location.
# Standard-library imports handle paths, serialized metadata, module lookup, and timing.
from pathlib import Path
import importlib
import json
import sys
import time

# Scientific packages provide tables, plotting, and the Astropy format expected by ParSNIP.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.table import Table

# Jupyter may start in the repository, package, or notebook directory, so search every parent.
cwd = Path.cwd().resolve()
search_roots = (cwd, *cwd.parents)
PROJECT_ROOT = next(
    (candidate for candidate in search_roots if (candidate / "warpTemplate" / "classification.py").exists()),
    None,
)
# Also recognize a launch from inside the warpTemplate Python package itself.
if PROJECT_ROOT is None:
    package_root = next(
        (candidate for candidate in search_roots if candidate.name == "warpTemplate" and (candidate / "classification.py").exists()),
        None,
    )
    PROJECT_ROOT = package_root.parent if package_root is not None else None
# Stop with the actual launch path instead of later failing with a misleading import error.
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate warp_templates above the Jupyter working directory: {cwd}")
# Resolve external dependencies from the kernel while hiding workspace reference copies.
def import_kernel_dependency(name, required_attributes=()):
    """Import one dependency without allowing the workspace root to shadow it."""

    original_sys_path = list(sys.path)
    kernel_sys_path = []
    for entry in original_sys_path:
        try:
            resolved_entry = Path(entry or Path.cwd()).resolve()
        except (OSError, RuntimeError):
            resolved_entry = None
        if resolved_entry != PROJECT_ROOT:
            kernel_sys_path.append(entry)
    for module_name in [key for key in sys.modules if key == name or key.startswith(f"{name}.")]:
        sys.modules.pop(module_name, None)
    try:
        sys.path[:] = kernel_sys_path
        module = importlib.import_module(name)
    finally:
        sys.path[:] = original_sys_path
    module_path = Path(module.__file__).resolve()
    missing = [attribute for attribute in required_attributes if not hasattr(module, attribute)]
    if PROJECT_ROOT in module_path.parents or missing:
        raise RuntimeError(
            "The active kernel must provide astronomical LSSTDESC ParSNIP "
            "(https://github.com/LSSTDESC/parsnip); "
            f"resolved {module_path}, missing APIs={missing}"
        )
    return module

parsnip = import_kernel_dependency(
    "parsnip", required_attributes=("ParsnipModel", "Classifier", "load_model")
)

# Prefer this checkout only for WarpTemplate itself.
sys.path.insert(0, str(PROJECT_ROOT))

from warpTemplate import classification as workflow

# Use one plot style throughout so cached and newly generated figures remain comparable.
sns.set_theme(context="notebook", style="ticks")

## Configuration

Run the audit first, then the smoke test. Completed smoke models, representations, classifiers, metrics, and figures are reused from their hashed result folder; full-training stages use completion markers in the same way. Keep every `FORCE_*` switch false unless you deliberately want to replace that cached stage. Only enable full training after reviewing the timing estimate. `EVALUATE_TEST` is a separate, explicit decision; once test predictions exist, replacement also requires `ALLOW_TEST_OVERWRITE = True`.

In [ ]:
# Define every choice that gives this experiment its identity and controls execution.

# --- Input data and leakage-safe split ---
# SAMPLE_ID names the immutable simulated sample. Changing it creates a different experiment.
SAMPLE_ID = "warp_sample_combined_schema6_56ecb71d1f32"
# SAMPLE_DIR is where the ensemble manifest and its 36 realization directories are read.
SAMPLE_DIR = PROJECT_ROOT / "training_samples" / SAMPLE_ID
# PACKAGE_ROOT contains reusable code plus generated split and classifier-run directories.
PACKAGE_ROOT = PROJECT_ROOT / "warpTemplate"
# template_key keeps identical warped templates together; basis_sn is stricter and also keeps related warps together.
SPLIT_STRATEGY = "basis_sn" #"template_key"  # Use "basis_sn" only as a separate, explicitly named experiment.
# The seed makes object selection, grouped folds, and stochastic training repeatable as far as the backend permits.
SEED = 20260721

# --- Compute budget and model-selection grid ---
# DEVICE tells PyTorch where ParSNIP tensors live. "cuda" needs an NVIDIA CUDA environment; this setup currently uses CPU.
DEVICE = "cpu"
# THREADS limits CPU parallelism used by ParSNIP/PyTorch; raising it can help until memory bandwidth becomes limiting.
THREADS = 14
# FULL_EPOCHS is the maximum number of full passes requested for ParSNIP, not an accuracy target.
# The current local fit loop records validation loss but does not stop automatically at its minimum.
FULL_EPOCHS = 100
# These are LightGBM settings, not ParSNIP settings. Larger values require more evidence before making small tree leaves.
MIN_CHILD_WEIGHTS = [1, 10, 30]

# --- Normal execution switches: enable expensive stages from top to bottom ---
# RUN_SMOKE executes or reloads the complete 64-objects-per-class miniature pipeline.
RUN_SMOKE = True
# RUN_FULL_TRAINING materializes all train/validation light curves and trains the full ParSNIP representation model.
RUN_FULL_TRAINING = False
# BUILD_REPRESENTATIONS applies a trained ParSNIP model to train, validation, and test objects and stores latent features.
BUILD_REPRESENTATIONS = False
# FIT_CLASSIFIER tunes LightGBM on validation features, then refits it using train plus validation features.
FIT_CLASSIFIER = False
# EVALUATE_TEST opens the official fold-0 test only after every model decision has been frozen.
EVALUATE_TEST = False
# Test results are intentionally immutable; set this only when deliberately replacing an already published test result.
ALLOW_TEST_OVERWRITE = False

# --- Cache override switches: normally keep every FORCE_* flag False ---
# A True value invalidates that completed cached stage and spends the corresponding compute again.
FORCE_RETRAIN_SMOKE = False  # Ignore and replace the completed miniature ParSNIP/LightGBM workflow.
FORCE_RETRAIN_FULL = False  # Ignore and replace the expensive full ParSNIP checkpoint.
FORCE_REBUILD_REPRESENTATIONS = False  # Recompute latent feature Parquets from the current ParSNIP model.
FORCE_REFIT_CLASSIFIER = False  # Repeat LightGBM selection/refit while leaving ParSNIP features unchanged.

# Apply the same seed before any sampling or model initialization.
workflow.set_random_seed(SEED)
# Split manifests are sample-specific; run products are additionally separated by split, backend, and configuration hash.
SPLIT_ROOT = PACKAGE_ROOT / "classification_splits" / SAMPLE_ID
RUNS_ROOT = PACKAGE_ROOT / "classifier_runs"
from parsnip.settings import default_settings as parsnip_default_settings

# The smoke folds are deliberately inside the ordinary training side of the experiment.
# Fold 0 therefore stays unseen even when the smoke workflow produces its own validation and confusion matrices.
SMOKE_CONFIG = {
    "epochs": 2,  # Two passes are enough to test learning and persistence, not to converge the model.
    "objects_per_class": 64,  # Equal counts make the smoke diagnostic insensitive to class frequency.
    "train_folds": list(range(4, 10)),  # Provenance groups available to smoke ParSNIP fitting.
    "validation_fold": 3,  # Used to inspect generalization and choose the LightGBM setting.
    "test_fold": 2,  # A disposable smoke-only test; this is not the official fold-0 test.
    "seed": SEED,  # Included in the cache identity because it changes sampled objects and initialization.
    "bands": list(workflow.EXPECTED_BANDS),  # Fixed order of all six LSST and three ZTF bands.
    "lightgbm_min_child_weight_grid": MIN_CHILD_WEIGHTS,  # Candidate values compared on smoke validation.
}
# The hash changes whenever a smoke-defining parameter changes, preventing incompatible cache reuse.
SMOKE_CACHE_ID = workflow.configuration_hash(SMOKE_CONFIG)

# The printed parsnip_settings below come from ParSNIP itself; this guide explains their roles.
# model_version: compatibility version of the serialized ParSNIP architecture/settings format.
# input_redshift: lets the encoder use z; predict_redshift: instead asks ParSNIP to infer z from photometry.
# specz_error: assumed uncertainty of known spectroscopic z, used only when redshift prediction is enabled.
# min_wave/max_wave: rest-frame wavelength range [Angstrom] represented by the generated spectrum.
# spectrum_bins: wavelength resolution of that spectrum; more bins increase detail, memory, and compute.
# max_redshift: numerical upper bound for a redshift predicted by ParSNIP, not the sample's actual z cut.
# band_oversampling: integration resolution for projecting spectra through filters; it must be odd.
# time_window: number of daily input-grid bins; observations outside this window do not enter the encoder.
# time_pad: extra phase range retained for prediction/reconstruction around the central encoder window.
# time_sigma/color_sigma: scale factors that convert normalized latent variables into days and color units.
# magsys/zeropoint: photometric convention; this adapter converts every observation consistently to AB zp=25.
# error_floor: minimum normalized flux uncertainty, preventing extremely precise points from dominating the loss.
# batch_size: light curves processed per optimizer step; larger batches need more memory.
# learning_rate: initial Adam step size; scheduler_factor multiplies it when progress stalls.
# min_learning_rate: fit stops once the scheduler pushes the learning rate below this threshold.
# penalty: strength of the spectral-smoothness regularization term in the ParSNIP loss.
# optimizer: update algorithm; sgd_momentum matters only if optimizer is changed from Adam to SGD.
# latent_size: number of intrinsic shape coordinates s1, s2, ... in addition to time, color, and amplitude.
# encode_*_architecture: layer widths in the encoder; conv dilations set its progressively wider time context.
# decode_architecture: layer widths used to turn phase plus latent coordinates into a generated spectrum.
# MODEL_CONFIG is persisted with the run so a saved checkpoint can be interpreted and reproduced later.
MODEL_CONFIG = {
    "max_epochs": FULL_EPOCHS,
    "threads": THREADS,
    "bands": list(workflow.EXPECTED_BANDS),
    # Copy all effective ParSNIP defaults instead of recording only settings changed in this notebook.
    "parsnip_settings": {key: value for key, value in parsnip_default_settings.items() if value is not None},
    "lightgbm_min_child_weight_grid": MIN_CHILD_WEIGHTS,
}
# ExperimentConfig defines comparability: sample, redshift assumption, split, backend, seed, and model choices.
CONFIG = workflow.ExperimentConfig(
    training_sample=SAMPLE_ID,  # Sample used to learn ParSNIP and LightGBM parameters.
    evaluation_sample=SAMPLE_ID,  # Sample whose frozen groups define the reported evaluation.
    backend="parsnip",
    redshift_mode="truth_z",  # Exact simulated z is supplied; treat this as an optimistic redshift baseline.
    split_strategy=SPLIT_STRATEGY,
    seed=SEED,
    model_config=MODEL_CONFIG,
)
# The normalized configuration determines RUN_DIR, so distinct scientific choices cannot silently share products.
RUN_DIR = workflow.run_directory(RUNS_ROOT, CONFIG)
MODEL_PATH = RUN_DIR / "models" / "parsnip.pt"
CLASSIFIER_PATH = RUN_DIR / "models" / "lightgbm.pkl"
FIGURE_DIR = RUN_DIR / "figures"
# Print the exact recorded configuration before any expensive stage starts.
print(json.dumps(CONFIG.normalized(), indent=2))
print(f"Outputs: {RUN_DIR}")

## Load, audit, and persist the split

Objects with fewer than three 0.33-day grouped observing epochs are excluded before splitting. The manifest is object-level and records both raw and merged labels, provenance IDs, fold, partition, strategy, and seed. Photometry remains only in the source observation tree.

In [ ]:
# Audit the schema-6 ensemble without loading 17 million observations at once.
# Truth has one row per simulated object; the observation audit streams the much larger photometry tree.
truth = workflow.load_sample_truth(SAMPLE_DIR)
source_manifest = workflow.load_sample_manifest(SAMPLE_DIR)
observation_audit = workflow.audit_sample_observations(SAMPLE_DIR)
# Merge only SLSN-I and SLSN-II; fitclass remains available as the unmodified raw label.
truth["final_label"] = workflow.merge_fitclasses(truth["fitclass"]).to_numpy()
print(f"Truth objects: {len(truth):,}; observation rows: {observation_audit['observation_rows']:,}")
print(f"Cadence realizations: {truth['survey_realization_id'].nunique()}")
display(pd.crosstab(truth["fitclass"], truth["final_label"], margins=True))
display(pd.Series(observation_audit["band_counts"], name="rows").reindex(workflow.EXPECTED_BANDS).to_frame())
# These assertions turn schema or band mismatches into an immediate failure before training.
assert set(truth["final_label"]) == set(workflow.FINAL_CLASSES)
assert set(observation_audit["band_counts"]) == set(workflow.EXPECTED_BANDS)
assert observation_audit["invalid_rows"] == 0

In [ ]:
# Create or validate both persistent group strategies, then activate the selected one.
# Existing manifests are reused; preparing both strategies makes later robustness comparisons deterministic.
split_manifests = {}
for strategy in ("template_key", "basis_sn"):
    split_manifests[strategy], excluded = workflow.prepare_persistent_split(
        SAMPLE_DIR, SPLIT_ROOT, strategy=strategy, seed=SEED
    )
# Only the selected strategy controls all downstream object membership in this run.
split_manifest = split_manifests[SPLIT_STRATEGY]
workflow.validate_split_manifest(split_manifest)
print(f"Excluded objects: {len(excluded)}")
display(excluded)
# Inspect both class balance and the number of indivisible provenance groups assigned to each partition.
display(pd.crosstab(split_manifest["final_label"], split_manifest["split"], margins=True))
display(
    pd.DataFrame({
        strategy: manifest.groupby("split")["group_id"].nunique()
        for strategy, manifest in split_manifests.items()
    }).fillna(0).astype(int)
)

## Training-only diagnostics

Cadence, signal-to-noise ratio (S/N), and example light curves are inspected only in the training partition. This is the same discipline as not looking at an exam before choosing how to study: test-set patterns must not influence model configuration.

In [ ]:
# Plot bounded training-only diagnostics without materializing the full ensemble.
# Diagnostics use training objects only, because even exploratory knowledge of test behavior can bias choices.
diagnostic_ids = workflow.sample_balanced_object_ids(
    split_manifest, per_class=512, seed=SEED + 10
)
# Load photometry only for the selected IDs; this keeps memory independent of the full 17-million-row tree.
training_observations = workflow.load_sample_observations(
    SAMPLE_DIR, object_ids=diagnostic_ids
)
# Reduce each light curve to cadence, epoch-count, SNR, and coverage summaries, then attach its class.
training_summary = workflow.summarize_light_curves(training_observations).merge(
    split_manifest[["object_id", "final_label"]], on="object_id"
)
# ECDFs show the complete distribution rather than hiding sparse tails behind a single mean.
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.ecdfplot(data=training_summary, x="median_cadence_days", hue="final_label", ax=axes[0])
sns.ecdfplot(data=training_summary, x="median_snr", hue="final_label", ax=axes[1], legend=False)
axes[0].set(xlabel="Median cadence [days]", xscale="log")
axes[1].set(xlabel="Median |flux| / flux error", xscale="log")
figure.tight_layout()
display(training_summary.groupby("final_label")[["observations", "epochs", "median_snr"]].median())

In [ ]:
# Display one representative training light curve per final class.
# Zeropoint conversion changes numerical flux units but preserves AB magnitude and signal-to-noise.
plot_observations = workflow.rescale_flux_to_zeropoint(training_observations)
# Choose the object nearest the class-median observation count, avoiding a subjective hand-picked example.
representative_ids = {}
for label, group in training_summary.groupby("final_label"):
    target_count = group["observations"].median()
    representative_ids[label] = group.iloc[(group["observations"] - target_count).abs().argmin()]["object_id"]
figure, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=False)
# Plot every survey band separately so cadence gaps and ZTF/LSST coverage remain visible.
for axis, (label, object_id) in zip(axes.flat, representative_ids.items()):
    light_curve = plot_observations[plot_observations["object_id"] == object_id]
    for band, band_rows in light_curve.groupby("band"):
        axis.errorbar(band_rows["mjd"], band_rows["flux"], yerr=band_rows["fluxerr"], fmt=".", label=band)
    axis.set(title=label, xlabel="MJD", ylabel="Flux (zeropoint 25)")
    axis.legend(ncol=3, fontsize=7)
for axis in axes.flat[len(representative_ids):]:
    axis.set_visible(False)
figure.tight_layout()

## Convert to `lcdata`

ParSNIP assumes fluxes use zeropoint 25. Each flux and uncertainty is multiplied by `10**((25 - zp) / 2.5)`, preserving AB magnitude and S/N. `mjd` becomes `time`, merged `fitclass` becomes `type`, and exact simulated `z` becomes `redshift`. All nine ZTF/LSST bands remain distinct.

In [ ]:
# Build full lcdata partitions only when full training is explicitly enabled.
# lcdata stores one in-memory table per object, so this is much more memory-intensive than the streaming audit.
train_dataset = None
validation_dataset = None
# Validation light curves are loaded separately and never participate in gradient updates.
if RUN_FULL_TRAINING:
    train_dataset = workflow.dataset_for_partition(truth, SAMPLE_DIR, split_manifest, "train")
    validation_dataset = workflow.dataset_for_partition(truth, SAMPLE_DIR, split_manifest, "validation")
    print(f"lcdata sizes: train={len(train_dataset):,}, validation={len(validation_dataset):,}")
    # A representative schema check catches unsorted times, invalid fluxes, or unknown band names early.
    first_curve = train_dataset.light_curves[0]
    assert np.all(np.diff(first_curve["time"]) >= 0)
    assert np.isfinite(first_curve["flux"]).all()
    assert set(first_curve["band"]).issubset(workflow.EXPECTED_BANDS)
else:
    # Keeping the full datasets as None prevents an accidental 100,000-object memory allocation during smoke work.
    print("Deferred full lcdata materialization; smoke subsets load directly from the ensemble.")

## Two-epoch smoke test and CPU timing

The smoke experiment is a complete miniature classifier workflow. It draws 64 objects per class from folds 4–9 for training, fold 3 for validation, and fold 2 for smoke testing. These remain provenance-group separated. ParSNIP trains for two epochs, LightGBM selects `min_child_weight` on smoke validation, and the refitted classifier produces validation and smoke-test confusion matrices. The official fold-0 test set remains untouched, so this result is for workflow verification rather than final scientific reporting. Because ParSNIP internally repeats augmented views for small datasets, it is still a meaningful CPU job. The measured wall time is scaled into a rough full-run estimate.

In [ ]:
# Run an end-to-end grouped smoke classifier with validation and test confusion matrices.
# This miniature run checks every interface; its two-epoch metrics are not final scientific performance.
smoke_history = None
smoke_output = RUN_DIR / "smoke" / f"smoke_{SMOKE_CACHE_ID}"
smoke_path = smoke_output / "parsnip_smoke.pt"
smoke_classifier_path = smoke_output / "lightgbm.pkl"
smoke_history_path = smoke_output / "history.json"
smoke_model_complete_path = smoke_output / "parsnip_complete.json"
smoke_complete_path = smoke_output / "complete.json"
smoke_validation_grid_path = smoke_output / "validation_grid.parquet"
smoke_validation_prediction_path = smoke_output / "validation_predictions.parquet"
smoke_test_prediction_path = smoke_output / "test_predictions.parquet"
smoke_validation_metric_path = smoke_output / "validation_metrics.json"
smoke_test_metric_path = smoke_output / "test_metrics.json"
smoke_figure_path = smoke_output / "confusion_matrices.png"
# A cache is complete only when the neural model, tabular classifier, predictions, metrics, and figure all exist.
smoke_cached_files = [
    smoke_path, smoke_classifier_path, smoke_history_path, smoke_model_complete_path, smoke_validation_grid_path,
    smoke_validation_prediction_path, smoke_test_prediction_path,
    smoke_validation_metric_path, smoke_test_metric_path, smoke_figure_path,
]
smoke_cache_complete = smoke_complete_path.exists() and all(path.exists() for path in smoke_cached_files)
# Removing only completion markers makes an intentional rerun explicit while leaving recoverable partial files visible.
if FORCE_RETRAIN_SMOKE and smoke_complete_path.exists():
    smoke_complete_path.unlink()
    smoke_cache_complete = False
if FORCE_RETRAIN_SMOKE and smoke_model_complete_path.exists():
    smoke_model_complete_path.unlink()
# The fast path reloads the complete result and performs no expensive neural-network work.
if RUN_SMOKE and smoke_cache_complete and not FORCE_RETRAIN_SMOKE:
    from IPython.display import Image

    smoke_history = json.loads(smoke_history_path.read_text())
    smoke_validation_grid = pd.read_parquet(smoke_validation_grid_path)
    smoke_validation_predictions = pd.read_parquet(smoke_validation_prediction_path)
    smoke_test_predictions = pd.read_parquet(smoke_test_prediction_path)
    smoke_validation_metrics = json.loads(smoke_validation_metric_path.read_text())
    smoke_test_metrics = json.loads(smoke_test_metric_path.read_text())
    print(f"Using completed smoke cache: {smoke_output}")
    display(Image(filename=str(smoke_figure_path)))
    display(smoke_validation_grid)
    display(
        pd.DataFrame(
            {
                "validation": {key: smoke_validation_metrics[key] for key in ("class_balanced_log_loss", "balanced_accuracy", "macro_f1", "top_1_accuracy", "top_2_accuracy")},
                "smoke_test": {key: smoke_test_metrics[key] for key in ("class_balanced_log_loss", "balanced_accuracy", "macro_f1", "top_1_accuracy", "top_2_accuracy")},
            }
        )
    )
    print(f"Cached ParSNIP training time: {smoke_history['elapsed_seconds'] / 60:.1f} min")
elif RUN_SMOKE:
    from sklearn.metrics import confusion_matrix

    # Select equal class counts from disjoint folds so class imbalance cannot dominate this systems check.
    smoke_train_ids = workflow.sample_balanced_object_ids(
        split_manifest, per_class=SMOKE_CONFIG["objects_per_class"], seed=SEED, folds=SMOKE_CONFIG["train_folds"]
    )
    smoke_validation_ids = workflow.sample_balanced_object_ids(
        split_manifest, per_class=SMOKE_CONFIG["objects_per_class"], seed=SEED + 1, folds=[SMOKE_CONFIG["validation_fold"]]
    )
    smoke_test_ids = workflow.sample_balanced_object_ids(
        split_manifest, per_class=SMOKE_CONFIG["objects_per_class"], seed=SEED + 2, folds=[SMOKE_CONFIG["test_fold"]]
    )
    # Check provenance groups, not merely object IDs: related simulated objects must not cross partitions.
    smoke_group_sets = {
        name: set(split_manifest.loc[split_manifest["object_id"].isin(ids), "group_id"])
        for name, ids in {"train": smoke_train_ids, "validation": smoke_validation_ids, "test": smoke_test_ids}.items()
    }
    assert not (smoke_group_sets["train"] & smoke_group_sets["validation"])
    assert not (smoke_group_sets["train"] & smoke_group_sets["test"])
    assert not (smoke_group_sets["validation"] & smoke_group_sets["test"])
    # Convert only the 3 x 448 selected objects into ParSNIP's lcdata representation.
    smoke_train_dataset = workflow.to_lcdata(truth, SAMPLE_DIR, smoke_train_ids)
    smoke_validation_dataset = workflow.to_lcdata(truth, SAMPLE_DIR, smoke_validation_ids)
    smoke_test_dataset = workflow.to_lcdata(truth, SAMPLE_DIR, smoke_test_ids)
    # A completed ParSNIP checkpoint can be reused even if a later LightGBM or plotting stage was interrupted.
    if smoke_path.exists() and smoke_history_path.exists() and smoke_model_complete_path.exists() and not FORCE_RETRAIN_SMOKE:
        reloaded_smoke = parsnip.load_model(str(smoke_path), device=DEVICE, threads=THREADS)
        smoke_history = json.loads(smoke_history_path.read_text())
        smoke_seconds = float(smoke_history["elapsed_seconds"])
        print(f"Reusing trained smoke ParSNIP checkpoint: {smoke_path}")
    else:
        smoke_model = parsnip.ParsnipModel(
            str(smoke_path), list(workflow.EXPECTED_BANDS), device=DEVICE, threads=THREADS
        )
        started = time.perf_counter()
        # Training data changes weights; validation data only measures the same ParSNIP loss after each epoch.
        # ParSNIP internally repeats very small datasets to process roughly 25,000 augmented examples per epoch.
        smoke_model.fit(
            smoke_train_dataset, max_epochs=SMOKE_CONFIG["epochs"], augment=True, test_dataset=smoke_validation_dataset
        )
        smoke_seconds = time.perf_counter() - started
        smoke_history = {"elapsed_seconds": smoke_seconds, "epochs": smoke_model.history}
        workflow.write_json_once(smoke_history, smoke_history_path, overwrite=True)
        workflow.write_json_once(
            {"status": "complete", "smoke_cache_id": SMOKE_CACHE_ID},
            smoke_model_complete_path, overwrite=True,
        )
        reloaded_smoke = parsnip.load_model(str(smoke_path), device=DEVICE, threads=THREADS)
    # A representation is the compact ParSNIP feature table consumed by the separate LightGBM classifier.
    smoke_representation_paths = {
        "smoke_train": smoke_output / "representations_train.parquet",
        "smoke_validation": smoke_output / "representations_validation.parquet",
        "smoke_test": smoke_output / "representations_test.parquet",
    }
    smoke_datasets = {
        "smoke_train": smoke_train_dataset,
        "smoke_validation": smoke_validation_dataset,
        "smoke_test": smoke_test_dataset,
    }
    smoke_representations = {}
    # Prediction of representations does not update ParSNIP; cached tables avoid repeating this inference stage.
    for split_name, representation_path in smoke_representation_paths.items():
        if representation_path.exists() and not FORCE_RETRAIN_SMOKE:
            smoke_representations[split_name] = Table.from_pandas(pd.read_parquet(representation_path))
            print(f"Using cached {split_name} representations")
        else:
            representation = reloaded_smoke.predict_dataset(smoke_datasets[split_name])
            smoke_representations[split_name] = representation
            workflow.write_table_once(
                representation.to_pandas(), representation_path, overwrite=FORCE_RETRAIN_SMOKE
            )
    # Train candidate LightGBM models on smoke_train and rank them only by smoke_validation log loss.
    selected_smoke_classifier, smoke_validation_grid = workflow.tune_parsnip_classifier(
        smoke_representations["smoke_train"],
        smoke_representations["smoke_validation"],
        MIN_CHILD_WEIGHTS,
    )
    best_smoke_weight = float(smoke_validation_grid.iloc[0]["min_child_weight"])
    raw_smoke_validation = selected_smoke_classifier.classify(
        smoke_representations["smoke_validation"]
    )
    # Once the hyperparameter is fixed, refit LightGBM with train plus validation before touching smoke_test.
    final_smoke_classifier = workflow.refit_parsnip_classifier(
        smoke_representations["smoke_train"],
        smoke_representations["smoke_validation"],
        best_smoke_weight,
    )
    final_smoke_classifier.write(str(smoke_classifier_path))
    reloaded_smoke_classifier = parsnip.Classifier.load(str(smoke_classifier_path))
    raw_smoke_test = reloaded_smoke_classifier.classify(smoke_representations["smoke_test"])

    # Build a smoke-specific manifest so every prediction records its exact role and provenance.
    smoke_manifest = split_manifest[
        split_manifest["object_id"].isin(
            smoke_train_ids + smoke_validation_ids + smoke_test_ids
        )
    ].copy()
    smoke_manifest["split"] = "smoke_train"
    smoke_manifest.loc[smoke_manifest["object_id"].isin(smoke_validation_ids), "split"] = "smoke_validation"
    smoke_manifest.loc[smoke_manifest["object_id"].isin(smoke_test_ids), "split"] = "smoke_test"
    smoke_validation_predictions = workflow.standardize_predictions(
        raw_smoke_validation, smoke_manifest, CONFIG, partition="smoke_validation"
    )
    smoke_test_predictions = workflow.standardize_predictions(
        raw_smoke_test, smoke_manifest, CONFIG, partition="smoke_test"
    )
    # Standard metrics evaluate class probabilities as well as the most likely class label.
    smoke_validation_metrics = workflow.compute_classification_metrics(
        smoke_validation_predictions
    )
    smoke_test_metrics = workflow.compute_classification_metrics(smoke_test_predictions)

    workflow.write_table_once(
        smoke_validation_grid, smoke_output / "validation_grid.parquet", overwrite=True
    )
    workflow.write_table_once(
        smoke_validation_predictions, smoke_output / "validation_predictions.parquet", overwrite=True
    )
    workflow.write_table_once(
        smoke_test_predictions, smoke_output / "test_predictions.parquet", overwrite=True
    )
    workflow.write_json_once(
        smoke_validation_metrics, smoke_output / "validation_metrics.json", overwrite=True
    )
    workflow.write_json_once(
        smoke_test_metrics, smoke_output / "test_metrics.json", overwrite=True
    )
    # This is only a first-order capacity estimate; small-dataset repeats make direct smoke scaling conservative.
    estimate = workflow.estimate_parsnip_training_time(
        smoke_seconds, len(smoke_train_ids), SMOKE_CONFIG["epochs"],
        int((split_manifest["split"] == "train").sum()), FULL_EPOCHS
    )

    # Row normalization turns each diagonal cell into recall for that true class.
    figure, axes = plt.subplots(1, 2, figsize=(15, 6))
    for axis, title, predictions in (
        (axes[0], "Smoke validation", smoke_validation_predictions),
        (axes[1], "Smoke test", smoke_test_predictions),
    ):
        matrix = confusion_matrix(
            predictions["true_class"],
            predictions["predicted_class"],
            labels=workflow.FINAL_CLASSES,
            normalize="true",
        )
        sns.heatmap(
            matrix, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues",
            xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axis,
        )
        axis.set(title=title, xlabel="Predicted class", ylabel="True class")
    figure.tight_layout()
    figure.savefig(smoke_figure_path, dpi=180, bbox_inches="tight")
    # Write the completion marker last: its presence certifies that all preceding smoke artifacts succeeded.
    workflow.write_json_once(
        {"status": "complete", "run_id": CONFIG.normalized()["run_id"], "smoke_cache_id": SMOKE_CACHE_ID, "smoke_config": SMOKE_CONFIG},
        smoke_complete_path, overwrite=True,
    )
    display(smoke_validation_grid)
    display(
        pd.DataFrame(
            {
                "validation": {key: smoke_validation_metrics[key] for key in ("class_balanced_log_loss", "balanced_accuracy", "macro_f1", "top_1_accuracy", "top_2_accuracy")},
                "smoke_test": {key: smoke_test_metrics[key] for key in ("class_balanced_log_loss", "balanced_accuracy", "macro_f1", "top_1_accuracy", "top_2_accuracy")},
            }
        )
    )
    print(f"Smoke time: {smoke_seconds / 60:.1f} min")
    print(f"First-order full CPU estimate: {estimate / 3600:.1f} h")
    print("The official fold-0 test set is still untouched.")
else:
    print("Set RUN_SMOKE=True for end-to-end training, validation, smoke testing, and confusion matrices.")

## Full ParSNIP training and representations

The full generative model sees training objects only and uses ParSNIP's built-in augmentation. Validation loss is diagnostic and may affect training duration, but test never participates. Representations are then generated for each partition. Do not inspect test representations until the classifier and experiment metadata have been frozen.

In [ ]:
# Run full ParSNIP training or reload the existing checkpoint.
# This stage learns the generative light-curve representation; it does not yet assign supernova classes.
model = None
FULL_HISTORY_PATH = RUN_DIR / "histories" / "parsnip_training.json"
FULL_COMPLETE_PATH = RUN_DIR / "models" / "parsnip_complete.json"
# Requiring model, history, and marker prevents a partially written checkpoint from looking complete.
full_cache_complete = MODEL_PATH.exists() and FULL_HISTORY_PATH.exists() and FULL_COMPLETE_PATH.exists()
if FORCE_RETRAIN_FULL and FULL_COMPLETE_PATH.exists():
    FULL_COMPLETE_PATH.unlink()
    full_cache_complete = False
# Reloading preserves the expensive result and allows later cells to continue in a fresh notebook kernel.
if RUN_FULL_TRAINING and full_cache_complete and not FORCE_RETRAIN_FULL:
    model = parsnip.load_model(str(MODEL_PATH), device=DEVICE, threads=THREADS)
    cached_training = json.loads(FULL_HISTORY_PATH.read_text())
    print(f"Using completed full-training cache: {MODEL_PATH}")
    print(f"Cached training time: {cached_training['elapsed_seconds'] / 3600:.2f} h")
elif RUN_FULL_TRAINING:
    # The validation dataset is scored after each epoch but is not used by the optimizer.
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    model = parsnip.ParsnipModel(
        str(MODEL_PATH), list(workflow.EXPECTED_BANDS), device=DEVICE, threads=THREADS
    )
    started = time.perf_counter()
    # Augmentation presents altered cadence/noise views; max_epochs is an upper training budget.
    model.fit(train_dataset, max_epochs=FULL_EPOCHS, augment=True, test_dataset=validation_dataset)
    training_seconds = time.perf_counter() - started
    workflow.write_json_once(
        {"elapsed_seconds": training_seconds, "epochs": model.history},
        FULL_HISTORY_PATH,
        overwrite=True,
    )
    workflow.write_json_once(
        {"status": "complete", "run_id": CONFIG.normalized()["run_id"], "model_config": MODEL_CONFIG},
        FULL_COMPLETE_PATH, overwrite=True,
    )
elif full_cache_complete:
    model = parsnip.load_model(str(MODEL_PATH), device=DEVICE, threads=THREADS)
    print(f"Reloaded {MODEL_PATH}")
elif MODEL_PATH.exists():
    # Do not trust a lone checkpoint after an interrupted write; completion metadata is mandatory.
    print(f"Found an incomplete full-training checkpoint and did not use it: {MODEL_PATH}")
else:
    print("No full checkpoint exists. Review the smoke timing, then set RUN_FULL_TRAINING=True.")

In [ ]:
# Generate and persist ParSNIP representations for all partitions without analyzing test.
# Representations contain latent coordinates and uncertainties, not final class probabilities.
representation_paths = {
    split: RUN_DIR / "representations" / f"{split}.parquet"
    for split in ("train", "validation", "test")
}
representations = {}
if BUILD_REPRESENTATIONS:
    # The same frozen ParSNIP checkpoint must encode every partition for a fair downstream comparison.
    if model is None:
        raise RuntimeError("Train or load a ParSNIP model before building representations")
    for split in ("train", "validation", "test"):
        if representation_paths[split].exists() and not FORCE_REBUILD_REPRESENTATIONS:
            representations[split] = Table.from_pandas(pd.read_parquet(representation_paths[split]))
            print(f"Using cached {split} representations")
        else:
            # Load one partition, disable stochastic augmentation, and persist its deterministic feature table.
            dataset = workflow.dataset_for_partition(truth, SAMPLE_DIR, split_manifest, split)
            table = model.predict_dataset(dataset, augment=False)
            representations[split] = table
            workflow.write_table_once(
                table.to_pandas(), representation_paths[split], overwrite=FORCE_REBUILD_REPRESENTATIONS
            )
else:
    # Even with building disabled, discover completed tables so later stages can resume from cache.
    for split, path in representation_paths.items():
        if path.exists():
            representations[split] = Table.from_pandas(pd.read_parquet(path))
    print(f"Available representations: {sorted(representations)}")

## Validation selection, refit, and freeze

LightGBM is the supervised classifier on ParSNIP's representation. Inverse-frequency weights make each class contribute equal total training weight. We compare `min_child_weight = 1, 10, 30` using class-balanced validation log loss, then refit that one choice on training plus validation. The untouched test set is still not scored.

In [ ]:
# Tune LightGBM on validation, refit on train plus validation, and freeze metadata.
# This is the supervised stage that maps ParSNIP features to the seven final labels.
classifier = None
validation_scores = None
VALIDATION_GRID_PATH = RUN_DIR / "metrics" / "validation_grid.parquet"
EXPERIMENT_PATH = RUN_DIR / "experiment.json"
CLASSIFIER_COMPLETE_PATH = RUN_DIR / "models" / "lightgbm_complete.json"
# A frozen experiment needs the classifier, validation comparison, metadata, and completion marker together.
classifier_cache_complete = all(
    path.exists() for path in (CLASSIFIER_PATH, VALIDATION_GRID_PATH, EXPERIMENT_PATH, CLASSIFIER_COMPLETE_PATH)
)
if FORCE_REFIT_CLASSIFIER and CLASSIFIER_COMPLETE_PATH.exists():
    CLASSIFIER_COMPLETE_PATH.unlink()
    classifier_cache_complete = False
if FIT_CLASSIFIER and classifier_cache_complete and not FORCE_REFIT_CLASSIFIER:
    classifier = parsnip.Classifier.load(str(CLASSIFIER_PATH))
    validation_scores = pd.read_parquet(VALIDATION_GRID_PATH)
    print(f"Using completed LightGBM cache: {CLASSIFIER_PATH}")
    display(validation_scores)
elif FIT_CLASSIFIER:
    # Validation features may choose a hyperparameter but official test features may not.
    if not {"train", "validation"}.issubset(representations):
        raise RuntimeError("Train and validation representations are required")
    # The returned table is sorted by class-balanced validation log loss, so row zero is selected.
    _, validation_scores = workflow.tune_parsnip_classifier(
        representations["train"], representations["validation"], MIN_CHILD_WEIGHTS
    )
    best_weight = float(validation_scores.iloc[0]["min_child_weight"])
    # Refit after selection so the final classifier can learn from validation without changing the chosen setting.
    classifier = workflow.refit_parsnip_classifier(
        representations["train"], representations["validation"], best_weight
    )
    classifier.write(str(CLASSIFIER_PATH))
    workflow.write_table_once(
        validation_scores, VALIDATION_GRID_PATH, overwrite=FORCE_REFIT_CLASSIFIER
    )
    # Mark the experiment frozen before test evaluation to document that all choices were already settled.
    frozen_metadata = workflow.build_experiment_metadata(CONFIG, split_manifest, status="frozen_before_test")
    frozen_metadata["selected_min_child_weight"] = best_weight
    workflow.write_json_once(
        frozen_metadata, EXPERIMENT_PATH, overwrite=FORCE_REFIT_CLASSIFIER
    )
    workflow.write_json_once(
        {"status": "complete", "run_id": CONFIG.normalized()["run_id"]},
        CLASSIFIER_COMPLETE_PATH, overwrite=True,
    )
    display(validation_scores)
elif classifier_cache_complete:
    classifier = parsnip.Classifier.load(str(CLASSIFIER_PATH))
    print(f"Reloaded {CLASSIFIER_PATH}")
elif CLASSIFIER_PATH.exists():
    print(f"Found an incomplete classifier cache and did not use it: {CLASSIFIER_PATH}")
else:
    print("Set FIT_CLASSIFIER=True after representations exist.")

## One-time test evaluation

Primary metrics are class-balanced log loss, balanced accuracy, and macro-F1. Per-class precision/recall/F1 exposes which classes fail; top-2 accuracy shows whether the correct class remains among plausible alternatives. The multiclass Brier score evaluates the full probability vector, while calibration compares stated confidence with empirical correctness. Confidence intervals resample the active provenance groups, not individual correlated light curves.

In [ ]:
# Evaluate the frozen classifier once and save standardized immutable test artifacts.
# This cell is a scientific release gate: test performance must not influence any earlier choice.
PREDICTION_PATH = RUN_DIR / "predictions" / "test.parquet"
TEST_METRIC_PATH = RUN_DIR / "metrics" / "test.json"
prediction_table = None
metrics = None
# Existing official results win by default, even when EVALUATE_TEST remains True on a later notebook run.
if EVALUATE_TEST and PREDICTION_PATH.exists() and TEST_METRIC_PATH.exists() and not ALLOW_TEST_OVERWRITE:
    prediction_table = pd.read_parquet(PREDICTION_PATH)
    metrics = json.loads(TEST_METRIC_PATH.read_text())
    print(f"Using frozen test cache: {PREDICTION_PATH}")
elif EVALUATE_TEST:
    # Refuse evaluation unless both the frozen classifier and fold-0 representations are present.
    if classifier is None or "test" not in representations:
        raise RuntimeError("A frozen classifier and test representations are required")
    raw_classifications = classifier.classify(representations["test"])
    prediction_table = workflow.standardize_predictions(
        raw_classifications, split_manifest, CONFIG, partition="test"
    )
    # Compute one global metric set, then quantify sampling uncertainty by resampling provenance groups.
    metrics = workflow.compute_classification_metrics(prediction_table)
    metrics["group_bootstrap_95"] = workflow.group_bootstrap_confidence_intervals(
        prediction_table, split_manifest, repeats=1000, seed=SEED
    )
    # Breakdowns reveal whether performance changes with redshift, cadence, SNR, or survey coverage.
    test_observations = workflow.select_partition_rows(SAMPLE_DIR, split_manifest, "test")
    breakdowns = workflow.metric_breakdowns(prediction_table, truth, test_observations)
    # All writes share the same explicit overwrite authorization to protect the frozen evaluation.
    workflow.write_table_once(prediction_table, PREDICTION_PATH, overwrite=ALLOW_TEST_OVERWRITE)
    workflow.write_json_once(metrics, TEST_METRIC_PATH, overwrite=ALLOW_TEST_OVERWRITE)
    workflow.write_table_once(breakdowns, RUN_DIR / "metrics" / "test_breakdowns.parquet", overwrite=ALLOW_TEST_OVERWRITE)
elif PREDICTION_PATH.exists():
    prediction_table = pd.read_parquet(PREDICTION_PATH)
    metrics = json.loads(TEST_METRIC_PATH.read_text())
else:
    print("Test is untouched. Set EVALUATE_TEST=True only after the experiment is frozen.")
if metrics is not None:
    display(pd.Series({key: value for key, value in metrics.items() if isinstance(value, (int, float))}))

## Post-evaluation analysis

These plots are intentionally gated on an existing frozen test prediction file. Predictive entropy is high when probability is spread across classes. Error tables should be read together with redshift, cadence, S/N, and survey coverage: they help identify failure regimes, but they must not be used to retune this already evaluated run. Any subsequent change is a new run ID.

In [ ]:
# Plot latent space, confusion matrices, calibration, uncertainty, and difficult cases.
# Nothing is plotted until official predictions exist, preventing accidental visual tuning on fold 0.
if prediction_table is not None:
    from sklearn.metrics import confusion_matrix

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    # Join latent coordinates to truth and predictions using object_id, never row position.
    test_representation = representations["test"].to_pandas()
    latent = test_representation.merge(
        prediction_table[["object_id", "true_class", "predicted_class"]], on="object_id"
    )
    probability_columns = [f"prob_{label}" for label in workflow.FINAL_CLASSES]
    probabilities = prediction_table[probability_columns].to_numpy()
    # Entropy is high for diffuse probabilities; clipping avoids log(0) without changing meaningful values.
    prediction_table["predictive_entropy"] = -(probabilities * np.log(np.clip(probabilities, 1e-15, 1))).sum(axis=1)

    figure, axes = plt.subplots(2, 2, figsize=(14, 11))
    sns.scatterplot(data=latent, x="s1", y="s2", hue="true_class", s=25, alpha=0.7, ax=axes[0, 0])
    # Raw counts expose sample size; row normalization makes the diagonal equal per-class recall.
    raw_confusion = confusion_matrix(prediction_table["true_class"], prediction_table["predicted_class"], labels=workflow.FINAL_CLASSES)
    normalized_confusion = confusion_matrix(prediction_table["true_class"], prediction_table["predicted_class"], labels=workflow.FINAL_CLASSES, normalize="true")
    sns.heatmap(raw_confusion, annot=True, fmt="d", xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[0, 1])
    sns.heatmap(normalized_confusion, annot=True, fmt=".2f", vmin=0, vmax=1, xticklabels=workflow.FINAL_CLASSES, yticklabels=workflow.FINAL_CLASSES, ax=axes[1, 0])
    # A calibrated model should lie near y=x: stated confidence then matches empirical accuracy.
    calibration = pd.DataFrame(metrics["calibration"])
    axes[1, 1].plot([0, 1], [0, 1], "k--", label="Ideal")
    axes[1, 1].plot(calibration["confidence"], calibration["accuracy"], "o-", label="Model")
    axes[1, 1].set(xlabel="Mean confidence", ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[1, 1].legend()
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "latent_confusion_calibration.png", dpi=180)

    # Rank wrong predictions by uncertainty and attach observing conditions for scientific error inspection.
    test_features = workflow.summarize_light_curves(workflow.select_partition_rows(SAMPLE_DIR, split_manifest, "test"))
    errors = prediction_table.merge(test_features, on="object_id")
    errors = errors[errors["true_class"] != errors["predicted_class"]].sort_values("predictive_entropy", ascending=False)
    display(errors[["object_id", "true_class", "predicted_class", "predictive_entropy", "median_snr", "median_cadence_days", "coverage"]].head(25))
else:
    print("Post-evaluation plots remain disabled until frozen test predictions exist.")

In [ ]:
# Inspect reconstructions for representative evaluated objects after test is frozen.
# These plots diagnose the generative ParSNIP model separately from LightGBM classification errors.
if prediction_table is not None and model is not None:
    test_dataset = workflow.dataset_for_partition(truth, SAMPLE_DIR, split_manifest, "test")
    # Key by object_id so reconstructed curves cannot be mismatched through changing table order.
    object_lookup = {curve.meta["object_id"]: curve for curve in test_dataset.light_curves}
    # Select deterministically rather than choosing visually attractive examples after seeing the curves.
    reconstruction_ids = prediction_table.sort_values("object_id").groupby("true_class").first()["object_id"].tolist()
    figure, axes = plt.subplots(4, 2, figsize=(14, 14))
    for axis, object_id in zip(axes.flat, reconstruction_ids):
        observed = object_lookup[object_id]
        # sample=False plots the posterior mean reconstruction rather than a random latent realization.
        model_times, model_flux, _ = model.predict_light_curve(observed, sample=False)
        for band in np.unique(observed["band"]):
            observed_band = observed[observed["band"] == band]
            axis.errorbar(observed_band["time"], observed_band["flux"], yerr=observed_band["fluxerr"], fmt=".", alpha=0.7)
            band_index = model.settings["bands"].index(band)
            axis.plot(model_times, model_flux[0, band_index], alpha=0.8, label=band)
        axis.set_title(object_id.rsplit(":", 2)[-2])
    for axis in axes.flat[len(reconstruction_ids):]:
        axis.set_visible(False)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / "representative_reconstructions.png", dpi=180)

## Next comparisons

Treat this exact-redshift ParSNIP run as one cell in a larger experiment matrix. Future samples and backends must reuse the same frozen evaluation groups. Runs are directly comparable only when both `evaluation_sample` and `split_strategy` match. A photometry-only ParSNIP design and the two SuperNNova redshift variants should receive new run IDs rather than changing this baseline. The complete SuperNNova implementation prompt is saved beside this notebook in `SUPERNNOVA_IMPLEMENTATION_PROMPT.md`.